# 05 — Fairness Audit

Runs the **Aequitas** fairness audit on CreditBridge model predictions.

Protected attributes checked:
- `gender` (M / F)
- `geography` (urban / semi-urban / rural)
- `income_proxy` (high / mid / low)

Fairness metrics:
- **Demographic Parity** — approval rates should not differ by > 10%
- **Equal Opportunity** — TPR (creditworthy correctly approved) should be consistent
- **FPR Parity** — non-creditworthy rejection rates should be consistent

> Assumes trained model: run `python src/model/train.py` first.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import json

from src.model.train import preprocess_features
from src.fairness.audit import run_fairness_audit
from sklearn.model_selection import train_test_split

MODEL_PATH = '../models/xgb_v1.pkl'
PROCESSED_PATH = '../data/processed/features.parquet'
FAIRNESS_REPORT_PATH = '../models/fairness_report.json'

payload = joblib.load(MODEL_PATH)
calibrated_model = payload['calibrated_model']
feature_names = payload['feature_names']

df = pd.read_parquet(PROCESSED_PATH)
df_prep = preprocess_features(df)
X = df_prep[feature_names]
y = df_prep['default_label']

# Recreate test split (must match training seed)
_, X_test, _, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
df_test_orig = df.iloc[X_test.index]

print(f'Test set: {X_test.shape[0]:,} applicants')
print(f'Default rate in test: {y_test.mean():.2%}')

## 1. Generate Model Predictions on Test Set

In [ ]:
y_prob = calibrated_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob > 0.50).astype(int)

print(f'Predicted default rate: {y_pred.mean():.2%}')
print(f'Approval rate (score threshold 0.5): {(1 - y_pred).mean():.2%}')

## 2. Run Aequitas Fairness Audit

In [ ]:
config = {
    'fairness': {
        'protected_attributes': ['gender', 'geography', 'income_proxy'],
        'reference_groups': {
            'gender': 'M',
            'geography': 'urban',
            'income_proxy': 'high',
        },
        'disparity_threshold': 0.10,
    }
}

try:
    report = run_fairness_audit(
        df_test_orig, y_test.values, y_pred,
        config, output_path=FAIRNESS_REPORT_PATH
    )
    print('\nAudit result:', 'PASSED ✓' if report['passed'] else 'FAILED ✗')
except Exception as e:
    print(f'Audit raised: {e}')
    with open(FAIRNESS_REPORT_PATH) as f:
        report = json.load(f)

## 3. Approval Rate by Group

In [ ]:
df_audit = df_test_orig.copy().reset_index(drop=True)
df_audit['predicted_default'] = y_pred
df_audit['approved'] = (y_pred == 0).astype(int)

for attr in ['gender', 'geography', 'income_proxy']:
    rates = df_audit.groupby(attr)['approved'].mean().sort_values(ascending=False)
    print(f'\nApproval rate by {attr}:')
    for k, v in rates.items():
        print(f'  {k:15s}: {v:.2%}')

## 4. Visualise Fairness Disparities

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, attr in zip(axes, ['gender', 'geography', 'income_proxy']):
    rates = df_audit.groupby(attr)['approved'].mean().sort_values(ascending=False)
    colors = ['#0066FF' if i == 0 else '#CCE0FF' for i in range(len(rates))]
    bars = ax.bar(rates.index, rates.values * 100, color=colors, edgecolor='#0A0A0A', linewidth=1.2)
    ax.axhline(y=rates.max() * 100 * 0.90, color='#D50000', linestyle='--', linewidth=1.2,
               label='10% disparity threshold')
    ax.set_title(f'Approval Rate by {attr.replace("_"," ").title()}', fontweight='bold')
    ax.set_ylabel('Approval rate (%)')
    ax.set_ylim(0, 100)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0f}%'))
    ax.legend(fontsize=8)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.5,
                f'{h:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Fairness Audit — Approval Rates by Protected Attribute', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('../models/fairness_approval_rates.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. True Positive Rate (Equal Opportunity) by Group

In [ ]:
# TPR = among actually solvent, fraction approved
df_audit['actual_solvent'] = (y_test.values == 0).astype(int)
df_audit['true_positive'] = df_audit['approved'] & df_audit['actual_solvent']

for attr in ['gender', 'geography', 'income_proxy']:
    tpr = df_audit[df_audit['actual_solvent'] == 1].groupby(attr)['approved'].mean()
    print(f'\nTPR (Equal Opportunity) by {attr}:')
    for k, v in tpr.items():
        print(f'  {k:15s}: {v:.2%}')

## 6. Audit Report Summary

In [ ]:
if os.path.exists(FAIRNESS_REPORT_PATH):
    with open(FAIRNESS_REPORT_PATH) as f:
        saved_report = json.load(f)
    
    print(f'Audit passed  : {saved_report["passed"]}')
    print(f'Violations    : {len(saved_report["violations"])}')
    print(f'Threshold     : {saved_report["disparity_threshold"]:.0%}')
    
    if saved_report['violations']:
        print('\nViolation details:')
        for v in saved_report['violations']:
            print(f'  [{v["group"]}={v["attribute"]}] {v["metric"]}: disparity={v["value"]:.3f}')
    else:
        print('\n✓ All groups within 10% disparity threshold.')
else:
    print('No saved report found.')

## 7. Recommendations

If any group shows FPR > 10% disparity:
- **Reweight** training data to oversample the underperforming group
- **Apply threshold calibration** per group (equalised odds post-processing)
- **Increase feature coverage** for rural applicants (GST signals have lower coverage)

Rural applicants consistently show the highest FPR — flagged for monitoring in the next model cycle (v1.4).